In [ ]:
Q. 1 — Driver or Executor?
Ans: 
A. Creates the Spark application. -> Driver
B. Builds the execution plan. -> Driver
C. Runs tasks on data partitions. -> Executor
D. Stores cached partition data. -> Executor
E. Coordinates work across executors -> Driver

In [ ]:
Q. 3 — Create a Local Spark Application 
Ans: 
from pyspark.sql import SparkSession
 
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("britmart_foundations")
    .getOrCreate()
)
 
data = [
    (101, "London", 45.50),
    (102, "Manchester", 18.00),
    (103, "Leeds", 30.00),
]
 
orders = spark.createDataFrame(
    data,
    ["order_id", "city", "amount_gbp"],
)
 
orders.show()
 
spark.stop()

In [ ]:
Q. 4 — What is the main job of the Spark driver? 
Ans: A. Coordinate the application and plan its work

In [ ]:
Q. 5 — Local or Cluster? 
Ans: 
A. Small demonstration on one laptop. -> Local
B. Production processing across many machines. -> Cluster
C. Testing basic syntax during training. -> Local
D. Shared processing of several terabytes. -> Cluster

In [ ]:
Q. 6 — Read a Local Master Setting
Explain: .master("local[4]")
local[4] runs Spark on one machine and allows up to four local worker threads.

It does not create four physical worker machines.

In [ ]:
Q. 7 — Create a complete Spark program using: 
Ans: 
from pyspark.sql import SparkSession
 
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("britmart_rdd_lab")
    .getOrCreate()
)
 
amounts = [
    45.50,
    18.00,
    62.25,
    12.00,
]
 
amounts_rdd = (
    spark.sparkContext
    .parallelize(amounts)
)
 
result = (
    amounts_rdd
    .filter(lambda amount: amount > 20)
    .map(lambda amount: round(amount * 1.10, 2))
    .collect()
)
 
print(result)
 
spark.stop() 

Result: [50.05, 68.48]

In [ ]:
Q. 8 — What does a Spark cluster manager mainly allocate? 
Ans: D. CPU and memory resources

In [ ]:
Q. 9 — Identify Transformations and Actions
Ans: 
filter() -> Transformation
select() -> Transformation
show() -> Action
count() -> Action
withColumn() -> Transformation
write.parquet() -> Action

In [ ]:
Q. 10 — Filter and Select
Create a complete Spark program with:
Ans: 
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
 
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("britmart_filter_lab")
    .getOrCreate()
)
 
data = [
    (101, "London", 45.50, "complete"),
    (102, "Manchester", 18.00, "cancelled"),
    (103, "London", 62.25, "complete"),
]
 
orders = spark.createDataFrame(
    data,
    [
        "order_id",
        "city",
        "amount_gbp",
        "status",
    ],
)
 
result = (
    orders
    .filter(col("status") == "complete")
    .select(
        "order_id",
        "city",
        "amount_gbp",
    )
)
 
result.show()
 
spark.stop()

In [ ]:
Q. 11 — Group and Aggregate
Create a complete Spark program using:
Ans: 
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as sum_
 
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("britmart_group_lab")
    .getOrCreate()
)
 
data = [
    (101, "London", 45.50),
    (102, "Leeds", 30.00),
    (103, "London", 62.25),
    (104, "Leeds", 20.00),
]
 
orders = spark.createDataFrame(
    data,
    ["order_id", "city", "amount_gbp"],
)
 
result = (
    orders
    .groupBy("city")
    .agg(
        sum_("amount_gbp")
        .alias("sales_gbp")
    )
)
 
result.show()
 
spark.stop()

Result: 
London -> 107.75
Leeds -> 50.00

In [ ]:
Q. 12 — Which operation triggers Spark execution? 
Ans:  B. show()

In [ ]:
Q. 13 — Explain Lazy Evaluation 
completed = orders.filter("status = 'complete'")
selected = completed.select("city", "amount_gbp")
Ans: 
Spark has recorded the transformations as part of a logical plan.

It normally waits for an action before executing the work.

This allows Spark to inspect and optimise the complete plan.

In [ ]:
Q. 14 — Job, Stage, Task, or Partition?
Match each term.
Ans: 
A. Work created when an action triggers execution. -> Job
B. A group of tasks separated by shuffle boundaries. -> Stage
C. One unit of work on one partition. -> Task
D. A logical slice of distributed data. -> Partition

In [ ]:
Q. 15 — Identify Shuffle Operations
Choose whether a shuffle is likely. 
Ans: 
filter rows by status -> usually no shuffle
select two columns -> usually no shuffle
groupBy city -> shuffle likely
join two large DataFrames by customer_id -> shuffle often likely
repartition by city -> shuffle

In [ ]:
Q. 16 — Which operation commonly causes a shuffle? 
Ans: C. groupBy("city")

In [ ]:
Q. 17 — Inspect a Spark Plan
Create a complete Spark program that:

In [ ]:

Ans: 
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as sum_
 
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("britmart_plan_lab")
    .getOrCreate()
)
 
data = [
    (101, "London", 45.50, "complete"),
    (102, "Manchester", 18.00, "cancelled"),
    (103, "London", 62.25, "complete"),
]
 
orders = spark.createDataFrame(
    data,
    [
        "order_id",
        "city",
        "amount_gbp",
        "status",
    ],
)
 
result = (
    orders
    .filter(col("status") == "complete")
    .groupBy("city")
    .agg(
        sum_("amount_gbp")
        .alias("completed_value_gbp")
    )
)
 
result.explain()
result.show()
 
spark.stop() 

explain() displays the plan. show() triggers computation for the displayed result.

In [ ]:
Q. 18 — Calculate Tasks from Partitions
A stage processes: 12 partitions
Assume one task is created per partition.

How many tasks are created in that stage?
Ans: 12 partitions
= 12 tasks

In [ ]:
Q. 19 — Find the Driver Risk 
A DataFrame contains: 50 million rows 
A developer runs: rows = df.collect()
Ans: 
collect() returns all rows to the driver.

With 50 million rows, the driver may run out of memory.

For large data, use distributed writes, aggregates, show(), or a small take() instead of collecting the entire dataset.

In [ ]:
Q. 20 — Compare Two Spark Pipelines
Pipeline A 
filter -> select -> show
Pipeline B 
groupBy -> aggregation -> show 
Ans: 
Pipeline B is more likely to contain a shuffle.

groupBy needs records with the same key to be brought together, which usually requires data movement between partitions.

Filtering and selecting can normally be completed inside each existing partition.